# Web Scraping für die Datenanalyse (Praxisprojekt)

### Lernziel
In diesem Projekt lernen wir, wie man strukturierte Daten von einer Website extrahiert, diese mit Pandas bereinigt und eine erste explorative Datenanalyse (EDA) durchführt.

**Der Workflow:**
1. **Extraktion:** HTML-Daten von `books.toscrape.com` abrufen.
2. **Parsing:** Mit BeautifulSoup gezielt Informationen aus dem HTML-Code filtern.
3. **Pagination:** Daten über mehrere Unterseiten hinweg sammeln.
4. **Bereinigung:** Datentypen mit Pandas korrigieren (z.B. Text zu Zahlen).
5. **Visualisierung:** Die Preisverteilung grafisch darstellen.

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import matplotlib.pyplot as plt
import time

# Einstellung für Grafiken im Notebook
%matplotlib inline
plt.style.use('ggplot')

### Schritt 1: Der Multi-Page Scraper
Wir nutzen eine `for`-Schleife für die ersten 5 Seiten. Ein `try-except`-Block stellt sicher, dass das Programm bei Verbindungsfehlern nicht abstürzt.

In [ ]:
base_url = "http://books.toscrape.com/catalogue/page-{}.html"
buecher_liste = []

for seite in range(1, 6):
    url = base_url.format(seite)
    
    try:
        antwort = requests.get(url)
        antwort.raise_for_status() # Prüft auf HTTP-Fehler (z.B. 404)
        soup = BeautifulSoup(antwort.text, "html.parser")
        
        # Alle Buch-Container finden
        artikel = soup.find_all("article", class_="product_pod")
        
        for buch in artikel:
            titel = buch.h3.a["title"]
            preis = buch.find("p", class_="price_color").text
            verfuegbarkeit = buch.find("p", class_="instock availability").text.strip()
            
            buecher_liste.append({
                "Titel": titel,
                "Preis": preis,
                "Status": verfuegbarkeit
            })
        
        print(f"Seite {seite} erfolgreich geladen.")
        time.sleep(1) # Höfliche Verzögerung, um den Server nicht zu überlasten
        
    except Exception as e:
        print(f"Fehler auf Seite {seite}: {e}")

df = pd.DataFrame(buecher_liste)
print(f"Fertig! Insgesamt {len(df)} Bücher gesammelt.")

### Schritt 2: Datenbereinigung
Scraping-Ergebnisse sind immer Texte (Strings). Wir müssen das Währungssymbol entfernen und den Preis in eine Dezimalzahl (Float) umwandeln.

In [ ]:
# Währungssymbol entfernen und Typ konvertieren
df['Preis'] = df['Preis'].str.replace('£', '', regex=False).astype(float)

df.head()

### Schritt 3: Analyse & Visualisierung
Wir berechnen statistische Kennzahlen und erstellen ein Histogramm.

In [ ]:
print(f"Durchschnittspreis: {df['Preis'].mean():.2f} £")
print(f"Teuerstes Buch: {df['Preis'].max():.2f} £")

plt.figure(figsize=(10, 6))
plt.hist(df['Preis'], bins=15, color='skyblue', edgecolor='black')
plt.title('Verteilung der Buchpreise')
plt.xlabel('Preis (£)')
plt.ylabel('Anzahl der Bücher')
plt.show()

### Schritt 4: Export
Speichern der bereinigten Daten als CSV-Datei.

In [ ]:
df.to_csv("buecher_daten.csv", index=False)
print("Daten wurden in 'buecher_daten.csv' gespeichert.")